<div dir="rtl">

# NB07-M01 — Tafsir Ingestion and Acceptance

## هدف
اجرای Fail-Closed Ingestion برای Tafsirهای تاییدشده.

این Notebook تا وقتی Source Manifest، Edition، Rights evidence و Structured records واقعی وجود نداشته باشند، هیچ متن Tafsir را Active نمی‌کند.

</div>

In [ ]:
# NB07-C01 — Imports and root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (p for p in [current, *current.parents] if (p / ".git").exists() and (p / "data").exists()),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, sha256_file, write_json_atomic
from qrag_phase1.tafsir import read_jsonl, validate_tafsir_rows, write_jsonl_atomic

assert collect_git_info(project_root)["branch"] == "qrag-phase1"

<div dir="rtl">

# NB07-M02 — Upstream Inventory Gate

NB06 باید Candidateهای Tafsir و Blockerهای Acquisition را ثبت کرده باشد.

</div>

In [ ]:
# NB07-C02 — Load upstream tafsir inventory acceptance
reports = project_root / "reports/phase1"
upstream_path = reports / "tafsir_inventory_acceptance_v1.json"

assert upstream_path.is_file(), "Run NB06 first."
upstream = json.loads(upstream_path.read_text(encoding="utf-8"))
assert upstream["objective_status"] == "ACHIEVED"

inventory = json.loads(
    (project_root / "data/phase1/manifests/tafsir_inventory_v1.json").read_text(encoding="utf-8")
)

<div dir="rtl">

# NB07-M03 — Discover Local Source Packages

فرمت مورد انتظار:
data/phase1/raw/tafsir/<tafsir_id-safe>/source_manifest.json
data/phase1/raw/tafsir/<tafsir_id-safe>/records.jsonl

وجود فایل کافی نیست؛ Rights و Validation هم باید PASS شوند.

</div>

In [ ]:
# NB07-C03 — Discover source packages
raw_root = project_root / "data/phase1/raw/tafsir"
package_results = []

for source in inventory["sources"]:
    safe_id = source["tafsir_id"].replace(":", "__")
    package_dir = raw_root / safe_id
    manifest_path = package_dir / "source_manifest.json"
    records_path = package_dir / "records.jsonl"

    package_results.append({
        "tafsir_id": source["tafsir_id"],
        "package_dir": str(package_dir.relative_to(project_root)),
        "manifest_exists": manifest_path.is_file(),
        "records_exists": records_path.is_file(),
        "manifest_path": manifest_path,
        "records_path": records_path,
    })

print(json.dumps([
    {k:v for k,v in item.items() if k not in {"manifest_path","records_path"}}
    for item in package_results
], ensure_ascii=False, indent=2))

<div dir="rtl">

# NB07-M04 — Validate Any Present Packages

Notebook می‌تواند در حالت صفر Package سالم اجرا شود و وضعیت BLOCKED_SOURCE_ACQUISITION بدهد؛ این رفتار Fail-Closed است.

</div>

In [ ]:
# NB07-C04 — Validate available packages
validated = []
blocked = []

for item in package_results:
    if not (item["manifest_exists"] and item["records_exists"]):
        blocked.append({
            "tafsir_id": item["tafsir_id"],
            "reason": "SOURCE_PACKAGE_MISSING",
        })
        continue

    manifest = json.loads(
        item["manifest_path"].read_text(encoding="utf-8")
    )

    if manifest["rights"]["status"] != "verified_allowed":
        blocked.append({
            "tafsir_id": item["tafsir_id"],
            "reason": "RIGHTS_NOT_VERIFIED",
        })
        continue

    rows = read_jsonl(item["records_path"])
    validation = validate_tafsir_rows(
        rows,
        item["tafsir_id"],
    )

    if validation["status"] != "PASS":
        blocked.append({
            "tafsir_id": item["tafsir_id"],
            "reason": "STRUCTURED_RECORD_VALIDATION_FAILED",
            "validation": validation,
        })
        continue

    validated.append({
        "tafsir_id": item["tafsir_id"],
        "manifest": manifest,
        "rows": rows,
        "records_path": item["records_path"],
    })

print("Validated packages:", len(validated))
print("Blocked packages:", len(blocked))

<div dir="rtl">

# NB07-M05 — Publish Only Validated Packages

هیچ Source Blocked وارد processed/tafsir نمی‌شود.

</div>

In [ ]:
# NB07-C05 — Publish validated tafsir artifacts
processed_root = project_root / "data/phase1/processed/tafsir"
artifact_rows = []

for item in validated:
    safe_id = item["tafsir_id"].replace(":", "__")
    output_path = processed_root / f"{safe_id}_v1.jsonl"

    write_jsonl_atomic(output_path, item["rows"])
    reloaded = read_jsonl(output_path)

    assert reloaded == item["rows"]

    artifact_rows.append({
        "tafsir_id": item["tafsir_id"],
        "path": str(output_path.relative_to(project_root)),
        "sha256": sha256_file(output_path),
        "record_count": len(reloaded),
        "status": "ACCEPTED",
    })

<div dir="rtl">

# NB07-M06 — Acceptance State

در وضعیت فعلی Repository انتظار داریم Sourceهای Tafsir هنوز Acquire نشده باشند. بنابراین Notebook باید صادقانه Blocked گزارش کند، نه اینکه داده نمونه را جای Source واقعی بنشاند.

</div>

In [ ]:
# NB07-C06 — Persist tafsir ingestion status
artifacts_dir = project_root / "artifacts/phase1/tafsir"
artifacts_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "schema_version": "qrag_tafsir_artifact_manifest_v1",
    "accepted_artifact_count": len(artifact_rows),
    "artifacts": artifact_rows,
    "blocked": blocked,
    "status": (
        "READY"
        if artifact_rows
        else "BLOCKED_SOURCE_ACQUISITION"
    ),
}
manifest_path = artifacts_dir / "tafsir_manifest_v1.json"
write_json_atomic(manifest_path, manifest)

acceptance = {
    "schema_version": "qrag_nb07_acceptance_v1",
    "objective_status": "FRAMEWORK_READY",
    "acceptance_status": (
        "PASS_WITH_LIMITATIONS"
        if artifact_rows
        else "BLOCKED"
    ),
    "accepted_tafsir_count": len(artifact_rows),
    "blocked_tafsir_count": len(blocked),
    "blocked": blocked,
    "next_action": (
        "CONTINUE_TO_DOWNSTREAM_SCHEMA"
        if artifact_rows
        else "ACQUIRE_AND_VERIFY_TAFSIR_SOURCE_PACKAGES"
    ),
}
acceptance_path = reports / "tafsir_ingestion_acceptance_v1.json"
write_json_atomic(acceptance_path, acceptance)

assert json.loads(manifest_path.read_text(encoding="utf-8")) == manifest
assert json.loads(acceptance_path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB07-M07 — جمع‌بندی

Framework Ingestion آماده است. برای ورود واقعی المیزان، تسنیم، نمونه، نورالثقلین یا البرهان باید Source Package واقعی طبق Contract فراهم و تأیید شود.

</div>